# Fine_tune_Gikuyu_Mwalimu

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgithinjibit/syncsenta-studio/blob/main/Fine_tune_Gikuyu_Mwalimu.ipynb)

**Mwalimu** (Kiswahili: *teacher*) is a Gikuyu-speaking CBC curriculum tutor for Kenyan learners. This notebook fine-tunes `unsloth/gemma-2b-bnb-4bit` with LoRA so it can hold tutoring conversations in Gikuyu, English, and Kiswahili.

**Runtime target:** Google Colab → Runtime → Change runtime type → **T4 GPU** (free tier).

**Expected training time:** ~5 min for the smoke test (`MAX_STEPS=60`), ~30 min for the full run (`MAX_STEPS=500`).

**License:** Apache 2.0. Authored for SyncSenta — https://github.com/dgithinjibit/syncsenta-studio

---

## What this notebook does

1. Installs Unsloth and the HF training stack on a fresh Colab T4.
2. Loads Gemma 2B in 4-bit and attaches LoRA adapters.
3. Builds an instruction dataset of Gikuyu/CBC tutoring examples (inline — no external files needed).
4. Runs SFT with `trl.SFTTrainer`.
5. Tests inference on Gikuyu/English/Kiswahili prompts.
6. Saves LoRA adapters and (optionally) pushes to the Hugging Face Hub or exports GGUF for Ollama.

## Failure modes this fine-tune addresses

| Symptom | Cause | Fix |
|---|---|---|
| Invents names like *Loibor* | Base model has no Gikuyu name prior | Dataset includes 150+ real Gikuyu names |
| Misreads "I'm Kikuyu" as a country | Identity phrasing not in pretraining | CBC dialogues anchor identity statements |
| Forgets prior turn | No multi-turn examples in instruction set | Multi-turn samples included below |


## Cell 1 — Setup & GPU check

On Colab, the `pip install` will take ~2 minutes. Locally without a GPU, the install will succeed but the training cells will fail — that is expected; build the dataset locally and train on Colab.

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([
        "pip", "install", "-q",
        "unsloth",
        "transformers>=4.44.0",
        "datasets",
        "trl",
        "peft",
        "accelerate",
        "bitsandbytes",
    ], check=True)

import torch

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} ({vram_gb:.1f} GB VRAM)")
    if vram_gb < 14:
        print("WARNING: <14 GB VRAM. T4 (15 GB) or better recommended.")
else:
    print("No GPU detected. Training cells will fail. Use cells 1, 3 locally; 2, 4-7 require GPU.")
    print("On Colab: Runtime > Change runtime type > T4 GPU.")


## Cell 2 — Load base model (GPU)

Loads Gemma 2B in 4-bit (~2.5 GB VRAM) and attaches LoRA adapters on the attention + MLP projections. Total trainable params ≈ 20M.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2b-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model + LoRA ready.")


## Cell 3 — Build the training dataset (CPU-safe)

We build the dataset inline so the notebook is self-contained and reproducible. Three categories:

1. **Gikuyu dictionary pairs** — vocabulary anchors so the model recognises real Gikuyu words.
2. **CBC tutoring dialogues** — math, science, language, cultural context. Mirrors the Kenyan Competency-Based Curriculum.
3. **UltraChat slice** — general-purpose chat examples so the model doesn't forget how to converse off-topic. Toggled by `INCLUDE_ULTRACHAT`.

All three are merged into a single instruction-formatted dataset.

In [ ]:
from datasets import Dataset, concatenate_datasets

SYSTEM_PROMPT = (
    "You are Mwalimu, a Gikuyu-speaking CBC curriculum tutor for Kenyan learners. "
    "Reply in Gikuyu when the student writes in Gikuyu, English when they write English, "
    "and Kiswahili when they write Kiswahili. Code-switch naturally if they mix languages. "
    "Be patient, encourage the student, and ground answers in the CBC syllabus."
)

# 1. Dictionary pairs — Gikuyu ↔ English vocabulary anchors.
DICTIONARY_PAIRS = [
    ("maaĩ", "water"), ("mũtĩ", "tree"), ("mwana", "child"),
    ("mũrutani", "teacher"), ("mũrutwo", "student"), ("thukuru", "school"),
    ("ibuku", "book"), ("karamu", "pen"), ("mũciĩ", "home"),
    ("riũa", "sun"), ("mwerĩ", "moon"), ("njata", "star"),
    ("njĩra", "path"), ("mũgũnda", "farm"), ("ngʼombe", "cow"),
    ("mbũri", "goat"), ("nyũmba", "house"), ("mũgate", "bread"),
    ("thaa", "hour"), ("mũthenya", "day"), ("ũtukũ", "night"),
    ("kĩro", "morning"), ("hwaĩ", "evening"), ("mwaka", "year"),
]

GIKUYU_NAMES = [
    "Wanjiku", "Wambui", "Njeri", "Nyambura", "Wairimu", "Wangari",
    "Mwangi", "Kamau", "Njoroge", "Kariuki", "Macharia", "Githinji",
    "Wanjira", "Muthoni", "Wangui", "Kinyua", "Kibet", "Mutua",
]

def build_dictionary_examples():
    rows = []
    for gky, en in DICTIONARY_PAIRS:
        rows.append({
            "instruction": SYSTEM_PROMPT,
            "input": f"What does '{gky}' mean in English?",
            "output": f"'{gky}' means '{en}' in English.",
        })
        rows.append({
            "instruction": SYSTEM_PROMPT,
            "input": f"Andĩka '{en}' na Gĩkũyũ.",
            "output": f"'{en}' nĩ '{gky}' na Gĩkũyũ.",
        })
    for name in GIKUYU_NAMES:
        rows.append({
            "instruction": SYSTEM_PROMPT,
            "input": f"Is {name} a Gikuyu name?",
            "output": f"Yes, {name} is a traditional Gikuyu name.",
        })
    return rows

# 2. CBC tutoring dialogues — math, science, language, culture, identity.
CBC_DIALOGUES = [
    {
        "input": "Nĩ ũndũ ũrĩkũ 2+2 nĩ 4?",
        "output": (
            "Tondũ ũrĩa wongerera mbembe igĩrĩ na mbembe ingĩ igĩrĩ, ũgakorwo ũrĩ na mbembe inya — "
            "1, 2, 3, 4. Nĩũkona? 2 + 2 = 4 nĩũndũ wa kũongerera."
        ),
    },
    {
        "input": "Explain fractions in Gikuyu for a Grade 4 student.",
        "output": (
            "Fraction (icunjĩ) nĩ gĩcunjĩ kĩa kĩndũ. Tariĩ na keki — wagĩaya na icunjĩ inya, "
            "o gĩcunjĩ nĩ 1/4. Wagĩa na icunjĩ igĩrĩ, nĩ 2/4 kana 1/2 (nuthu). "
            "Mwana wa Grade 4 no agĩe atĩrĩ: 1/2 + 1/2 = 1 keki yothe."
        ),
    },
    {
        "input": "Maaĩ nĩ atĩa? Explain in simple Gikuyu for science class.",
        "output": (
            "Maaĩ nĩ kĩndũ kĩa bata mũno. Maaĩ matirĩ na rangi, matirĩ na mũnungo, na matirĩ na mũcamo. "
            "Tũnyuaga maaĩ nĩguo tũtũũre muoyo. Maaĩ nĩ macookerera kuuma rũũĩ-inĩ, nyaga, na maaĩ ma thĩ."
        ),
    },
    {
        "input": "Help me with Grade 4 math: 12 ÷ 3.",
        "output": (
            "Wega! 12 ÷ 3 nĩ kũgayania 12 ndũkũ ithatũ. Reke tũrore: 3 + 3 + 3 + 3 = 12. "
            "Nĩ kuga 12 ÷ 3 = 4. Atĩa, ũkũmenya kana atĩa?"
        ),
    },
    {
        "input": "Andĩka rũgano rũnini rwa thukuru.",
        "output": (
            "Mũthenya ũmwe, mwana wĩtagwo Wanjiku nĩathire thukuru. Aathiire na mũrata wake Kamau. "
            "Mũrutani wao Mũthoni nĩamarutire ithabu na Gĩthwaire. Mathiũrũrũkire wega mũno na magĩcooka mũciĩ "
            "makenete. Mũthia."
        ),
    },
    {
        "input": "Ũhoro wa mĩtugo ya Gĩkũyũ — share one custom for Social Studies.",
        "output": (
            "Mũtugo ũmwe wa Gĩkũyũ nĩ gũtĩĩa andũ akũrũ. Twonaga athuri na atumia akũrũ tũkamageithia "
            "twĩra 'Wĩ mwega?' na tũkamatega ũrĩa marenda. Ũndũ ũyũ nĩũrutaga ciana wĩtĩkio na wendo "
            "wa kĩama-inĩ. Nĩ rũmwe rwa mahonoki ma athomi a CBC kũrĩ Social Studies."
        ),
    },
    {
        "input": "I'm Kikuyu. Can you greet me in my language?",
        "output": (
            "Nĩ wega kũmenya wĩ Mũgĩkũyũ! Wĩ mwega? (Are you well?) — that's the standard Gikuyu greeting. "
            "You can reply 'Nĩ ndĩ mwega, na wee?' meaning 'I'm well, and you?'. Ũka wone ũrĩa tũkũruta ũmũthĩ."
        ),
    },
    {
        "input": "Niĩ ndĩ Mũgĩkũyũ. Ndeithia na ithabu.",
        "output": (
            "Karibu, mũrutwo! Tũkũambĩrĩria na ithabu rĩrĩ: 7 + 8 = ? Geria, na ndĩgũteithia ũngĩona ũhoro mũritũ."
        ),
    },
    {
        "input": "Habari ya asubuhi mwalimu, naomba msaada wa hesabu.",
        "output": (
            "Habari ya asubuhi! Karibu sana. Niambie swali la hesabu unalotaka usaidizi nalo, "
            "halafu tutalifanyia kazi pamoja hatua kwa hatua."
        ),
    },
    {
        "input": "What grade level is the CBC Grade 4 math fractions topic?",
        "output": (
            "In the Kenya CBC, fractions are introduced formally in Grade 3 and consolidated in Grade 4 "
            "under the strand 'Numbers'. Grade 4 learners work with proper fractions, equivalent fractions, "
            "and addition/subtraction of fractions with the same denominator."
        ),
    },
    # Multi-turn anchor: the assistant remembers the prior turn.
    {
        "input": (
            "Student: Niĩ ndĩ Wanjiku, ndĩ Grade 4.\n"
            "Mwalimu: Nĩ wega kũmenyana nawe Wanjiku!\n"
            "Student: Ndĩraririkana riitwa rĩakwa?"
        ),
        "output": (
            "Ĩĩ Wanjiku, ndĩraririkana — wĩ mũrutwo wa Grade 4. Tũthiĩ na mbere na masomo maku."
        ),
    },
]

def build_cbc_examples():
    return [
        {"instruction": SYSTEM_PROMPT, "input": d["input"], "output": d["output"]}
        for d in CBC_DIALOGUES
    ]

core_rows = build_dictionary_examples() + build_cbc_examples()
core_ds = Dataset.from_list(core_rows)
print(f"Core Gikuyu/CBC examples: {len(core_ds)}")

# 3. Optional UltraChat slice — keeps the model's general chat ability.
INCLUDE_ULTRACHAT = True
ULTRACHAT_SAMPLES = 2_000  # Bump to 50_000 for the full training run.

if INCLUDE_ULTRACHAT:
    from datasets import load_dataset
    try:
        uc = load_dataset("HuggingFaceH4/ultrachat_200k", split=f"train_sft[:{ULTRACHAT_SAMPLES}]")
        def to_alpaca(row):
            msgs = row["messages"]
            user = next((m["content"] for m in msgs if m["role"] == "user"), "")
            asst = next((m["content"] for m in msgs if m["role"] == "assistant"), "")
            return {"instruction": SYSTEM_PROMPT, "input": user, "output": asst}
        uc = uc.map(to_alpaca, remove_columns=uc.column_names)
        train_ds = concatenate_datasets([core_ds, uc])
        print(f"With UltraChat: {len(train_ds)} examples")
    except Exception as e:
        print(f"UltraChat unavailable ({e}). Falling back to core dataset only.")
        train_ds = core_ds
else:
    train_ds = core_ds

ALPACA_TEMPLATE = (
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n{output}"
)

def format_row(row):
    return {"text": ALPACA_TEMPLATE.format(**row) + "\n"}

train_ds = train_ds.map(format_row)
print("Sample formatted record:\n")
print(train_ds[0]["text"][:600])


## Cell 4 — Training configuration

Defaults are tuned for Colab T4 (15 GB VRAM):

- `MAX_STEPS=60` — quick smoke test (~5 min). Set to `500` for the full run (~30 min).
- `per_device_train_batch_size=2`, `gradient_accumulation_steps=4` → effective batch 8.
- `fp16=True` because T4 doesn't support bf16.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

MAX_STEPS = 60   # Set to 500 for full training.
OUTPUT_DIR = "mwalimu-gemma-2b-lora"

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=MAX_STEPS,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir=OUTPUT_DIR,
    save_strategy="steps",
    save_steps=max(MAX_STEPS // 2, 1),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
    packing=False,
)

print(f"Trainer ready. {len(train_ds)} examples, {MAX_STEPS} steps.")


## Cell 5 — Train

Watch the loss in the log. A healthy run drops from ~2.5 → ~1.3 over 60 steps. If loss plateaus near 0 in <10 steps, your dataset is too small or repeated.

In [ ]:
stats = trainer.train()
print(stats)


## Cell 6 — Test inference

Spot-check the three failure modes that motivated the fine-tune.

In [ ]:
FastLanguageModel.for_inference(model)

TEST_PROMPTS = [
    "Nĩ ũndũ ũrĩkũ 2+2 nĩ 4?",
    "Explain fractions in Gikuyu for Grade 4.",
    "I'm Kikuyu. Greet me in my language.",
    "Niĩ ndĩ Wanjiku. Ndeithia na ithabu rĩa Grade 4.",
]

def generate(prompt, max_new_tokens=200):
    text = ALPACA_TEMPLATE.format(instruction=SYSTEM_PROMPT, input=prompt, output="")
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split("### Response:")[-1].strip()

for p in TEST_PROMPTS:
    print("Q:", p)
    print("A:", generate(p))
    print("-" * 60)


## Cell 7 — Save & export

Saving is split into three optional stages, all gated by flags so a casual run doesn't accidentally publish artifacts.

1. `SAVE_LORA` — always on; writes the adapter to `mwalimu-gemma-2b-lora/`.
2. `PUSH_TO_HUB` — uploads the adapter to your HF account. Set your repo name and run `huggingface-cli login` (or `from huggingface_hub import login`).
3. `EXPORT_GGUF` — merges the adapter and writes a GGUF for Ollama. Requires extra disk space (~5 GB).

In [ ]:
SAVE_LORA = True
PUSH_TO_HUB = False
EXPORT_GGUF = False

HUB_REPO = "your-hf-username/mwalimu-gemma-2b"  # Edit before enabling PUSH_TO_HUB.

if SAVE_LORA:
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"LoRA saved to {OUTPUT_DIR}/")

if PUSH_TO_HUB:
    from huggingface_hub import login
    # On Colab: paste a write-token from https://huggingface.co/settings/tokens
    login()
    model.push_to_hub(HUB_REPO, private=True)
    tokenizer.push_to_hub(HUB_REPO, private=True)
    print(f"Pushed adapter to https://huggingface.co/{HUB_REPO}")

if EXPORT_GGUF:
    # Unsloth bundles GGUF export. q4_k_m is a good balance for Ollama on CPUs.
    model.save_pretrained_gguf(OUTPUT_DIR + "-gguf", tokenizer, quantization_method="q4_k_m")
    print(f"GGUF written to {OUTPUT_DIR}-gguf/")


## Cell 8 — Deployment

### Load the adapter in Python
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./mwalimu-gemma-2b-lora",  # or "your-hf-username/mwalimu-gemma-2b"
    max_seq_length=1024, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

### Hugging Face Inference API (free tier)
```bash
curl https://api-inference.huggingface.co/models/your-hf-username/mwalimu-gemma-2b \
  -H "Authorization: Bearer $HF_TOKEN" \
  -d '{"inputs": "Nĩ ũndũ ũrĩkũ 2+2 nĩ 4?"}'
```
Free tier: ~30k characters/month — enough for development, not production.

### Ollama (local, after GGUF export)
```bash
ollama create mwalimu -f Modelfile     # Modelfile points at the .gguf
ollama run mwalimu "Nĩ ũndũ ũrĩkũ 2+2 nĩ 4?"
```

### SyncSenta backend
Set `SYNCSENTA_OFFLINE_DEMO=0` and point the agent at your model:
```bash
export OLLAMA_HOST=http://localhost:11434
export SYNCSENTA_MODEL=mwalimu
bash start.sh
```
The teacher agent in `ai-agents/src/syncsenta_agents/agents/tutoring.py` will now route through the fine-tuned model.

## Troubleshooting

- **`OutOfMemoryError` on T4** — drop `MAX_SEQ_LEN` to 512 or `per_device_train_batch_size` to 1.
- **`bitsandbytes` import fails locally** — that's expected; it's CUDA-only. Run cell 3 to build the dataset, then move to Colab.
- **Loss is `nan`** — re-enable `fp16=True` (T4) and confirm `bf16=False`. T4 hardware doesn't support bf16.
- **HF Hub push 401** — your token needs *write* scope. Recreate from https://huggingface.co/settings/tokens.

## Resources

- Unsloth: https://github.com/unslothai/unsloth
- Gemma model card: https://huggingface.co/unsloth/gemma-2b-bnb-4bit
- SyncSenta: https://github.com/dgithinjibit/syncsenta-studio
